In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim


from torchvision import datasets,transforms
from torch.utils.data import Dataset,DataLoader


transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])


train_data = datasets.MNIST(root='data', train=True, download=True, transform=transform)
test_data = datasets.MNIST(root='data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data , batch_size=64 , shuffle=True)
test_loader = DataLoader(test_data , batch_size=64 , shuffle=False)



Select Device (GPU or CPU)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device =", device)

Define the CNN Model for MNIST Classification

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layers=nn.Sequential(
            nn.Conv2d(1,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )
        self.fc_layers=nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7,128),
            nn.ReLU(),
            nn.Linear(128,10)
        )
    def forward(self,x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x
model = CNN().to(device)

        

Define Loss Function and Optimizer

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.001)

Training Loop for the CNN Model

In [ ]:
loss_history = []
epochs=5
for epoch in range(epochs):
    total_loss = 0
    for images,labels in train_loader:
        images,labels=images.to(device),labels.to(device)
        outputs=model(images)
        loss=criterion(outputs,labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
    loss_history.append(total_loss)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4}")

Evaluate CNN Model Accuracy on MNIST Test Set

In [ ]:
correct = 0
total = 0
with torch.no_grad():
    for images,labels in test_loader:
        images,labels=images.to(device),labels.to(device)
        outputs=model(images)
        _,predicted = torch.max(outputs.data,1)
        total+=labels.size(0)
        correct+=(predicted==labels).sum().item()
print("Accuracy=",correct/total)

Plot Training Loss Curve (CNN Model)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(loss_history)
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

Load & Preprocess External Image for CNN Prediction

In [ ]:
from PIL import Image
img=Image.open("5.png") #choose any other pic you want or have
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((28,28)),
    transforms.ToTensor(),
    transforms.Normalize((0.5),(0.5))
])
img=transform(img)
img=img.unsqueeze(0)
img=img.to(device)

Predict Digit from External Image Using CNN

In [ ]:
model.eval()
with torch.no_grad():
    output=model(img)
    _,predicted=torch.max(output,1)
    print("Predicted digit: ",predicted.item())

Display the Preprocessed Input Image

In [ ]:
img_show=img.cpu().squeeze()*0.5+0.5
plt.imshow(img_show,cmap='gray')
plt.title("Input Image")
plt.show